# 07 — Video Demo: Soccer Match Homography Visualization

Notebook ini mengambil **model terbaik dari hasil benchmark** dan menerapkannya ke video pertandingan sepak bola frame-by-frame.

**Output:** Video `.mp4` side-by-side:
- **Kiri:** frame asli kamera broadcast + overlay keypoint terdeteksi (hijau = visible, oranye = occluded)
- **Kanan:** peta taktis bird's-eye view dengan proyeksi posisi keypoint

**Runtime:** T4 GPU. Estimasi ~1–2 menit per menit video.

**Prereq:** Notebook `03_evaluation.ipynb` sudah selesai dan `results_master.csv` tersedia.

---
**Cara pakai:**
1. Jalankan semua cell berurutan
2. Cell 4: isi path video input kamu
3. Cell 5: pilih model (atau biarkan auto-select model terbaik dari CSV)
4. Cell 7: jalankan proses video
5. Cell 9: download hasilnya

In [ ]:
# Cell 1: Install dependencies
!pip install ultralytics opencv-python-headless numpy pandas pyyaml tqdm matplotlib --quiet
# Uncomment jika model terbaik dari keluarga HRNet/ViTPose:
# !pip install -U openmim --quiet
# !mim install mmengine mmcv mmpose --quiet
print('Dependencies ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Environment & Path Setup — kompatibel Kaggle DAN Colab (auto-detect)
# ══════════════════════════════════════════════════════════════════════
import os, sys

def _detect_env():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

ENV = _detect_env()
print(f'Environment terdeteksi: {ENV}')

# ⚠️ GANTI dengan URL repo GitHub kamu sendiri (isi SEKALI saja per notebook)
GITHUB_REPO = 'https://github.com/atangorp/soccer-homography-benchmark.git'

def _link(src, dst):
    """Symlink src -> dst kalau src ada dan dst belum ada. Aman dipanggil berkali-kali."""
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)

if ENV == 'kaggle':
    PROJECT_ROOT = '/kaggle/working/soccer-homography-benchmark'
    if not os.path.exists(PROJECT_ROOT):
        os.system(f'git clone -q {GITHUB_REPO} "{PROJECT_ROOT}"')
    sys.path.insert(0, PROJECT_ROOT)

    # WAJIB: attach dataset "soccer-field-raw" via "+ Add Input" di sidebar kanan
    # (ganti 'soccer-field-raw' kalau slug dataset kamu berbeda)
    _link('/kaggle/input/soccer-field-raw',
          f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8')
    # Weights hasil training — attach masing-masing notebook output via "+ Add Input"
    _link('/kaggle/input/02a-train-yolo11/soccer-homography-benchmark/artifacts/weights/yolo11',
          f'{PROJECT_ROOT}/artifacts/weights/yolo11')
    _link('/kaggle/input/02b-train-hrnet/soccer-homography-benchmark/artifacts/weights/hrnet',
          f'{PROJECT_ROOT}/artifacts/weights/hrnet')
    _link('/kaggle/input/02c-train-vitpose/soccer-homography-benchmark/artifacts/weights/vitpose',
          f'{PROJECT_ROOT}/artifacts/weights/vitpose')
    _link('/kaggle/input/02d-train-detr/soccer-homography-benchmark/artifacts/weights/detr',
          f'{PROJECT_ROOT}/artifacts/weights/detr')
    # Gambar test set (dibutuhkan evaluasi visual + GT label)
    for _split in ['train', 'val', 'test']:
        _link(f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8/{_split}/images',
              f'{PROJECT_ROOT}/data/converted/images/{_split}')
    _link('/kaggle/input/03-evaluation/soccer-homography-benchmark/artifacts/logs/evaluation/results_master.csv',
          f'{PROJECT_ROOT}/artifacts/logs/evaluation/results_master.csv')
    # Video pertandingan kamu sendiri — attach dataset video kamu via "+ Add Input"
    # (ganti 'my-match-video' dengan slug dataset video kamu)

elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/soccer-homography-benchmark'
    sys.path.insert(0, PROJECT_ROOT)

else:
    PROJECT_ROOT = os.getcwd()
    sys.path.insert(0, PROJECT_ROOT)

import json, yaml, time
import numpy as np, pandas as pd, cv2, torch
from tqdm import tqdm
from copy import deepcopy

with open(f'{PROJECT_ROOT}/configs/eval_global.yaml') as f:
    CFG = yaml.safe_load(f)

RESULTS_CSV   = f'{PROJECT_ROOT}/artifacts/logs/evaluation/results_master.csv'
VIDEO_OUT_DIR = f'{PROJECT_ROOT}/artifacts/results/videos'
os.makedirs(VIDEO_OUT_DIR, exist_ok=True)

print(f'Results CSV : {"OK" if os.path.exists(RESULTS_CSV) else "belum ada — jalankan 03 dulu"}')
print(f'GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'PROJECT_ROOT : {PROJECT_ROOT}')


In [ ]:
# Cell 3: Auto-select model terbaik
if os.path.exists(RESULTS_CSV):
    df_r = pd.read_csv(RESULTS_CSV)
    df_r = df_r.sort_values(['detection_rate','mpe_mean'], ascending=[False,True]).reset_index(drop=True)
    print('Top 5 model berdasarkan detection rate + MPE:')
    print(df_r[['model_full_name','detection_rate','mpe_mean','fps']].head(5).to_string(index=False))
    BEST_FAMILY  = df_r.iloc[0]['model_family']
    BEST_VARIANT = df_r.iloc[0]['model_variant']
    BEST_NAME    = df_r.iloc[0]['model_full_name']
    print(f'\nModel terpilih: {BEST_NAME}')
else:
    BEST_FAMILY, BEST_VARIANT, BEST_NAME = 'yolo11', 'small', 'YOLO11-Small'
    print(f'Fallback ke default: {BEST_NAME}')

In [ ]:
# Cell 4: Konfigurasi video input
# ── Ganti path ini dengan video pertandingan kamu ──────────────────────────
VIDEO_INPUT = '/content/drive/MyDrive/Colab Notebooks/Videos/match_sample.mp4'

# Konfigurasi proses
PROCESS_FPS   = 5      # Frame/detik yg diproses (None = semua frame)
MAX_SECONDS   = None   # Batasi durasi (None = keseluruhan video)
OUTPUT_SCALE  = 0.75   # Skala resolusi output (0.75 = 75% dari asli)
SHOW_KPT_ID   = True   # Tampilkan nomor ID keypoint
SHOW_STATS    = True   # Tampilkan statistik real-time
# ───────────────────────────────────────────────────────────────────────────

if os.path.exists(VIDEO_INPUT):
    cap = cv2.VideoCapture(VIDEO_INPUT)
    fps_orig  = cap.get(cv2.CAP_PROP_FPS)
    n_frames  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    vid_w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    vid_h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    dur = n_frames / fps_orig
    print(f'Video  : {os.path.basename(VIDEO_INPUT)}')
    print(f'Ukuran : {vid_w}x{vid_h} @ {fps_orig:.1f} fps')
    print(f'Durasi : {dur:.1f}s ({n_frames} frames)')
    step = max(1, int(fps_orig / PROCESS_FPS)) if PROCESS_FPS else 1
    n_proc = len(range(0, n_frames, step))
    print(f'Frames yg diproses: ~{n_proc} (1 per {step} frame)')
else:
    print(f'VIDEO TIDAK DITEMUKAN: {VIDEO_INPUT}')
    print('Upload video ke Google Drive dulu, lalu ganti path di atas')

In [ ]:
# Cell 5: Load model
W_BASE = f'{PROJECT_ROOT}/artifacts/weights'

def load_model_for_demo(family, variant):
    conf = CFG['conf_threshold']
    if family == 'yolo11':
        from src.models.yolo11_wrapper import YOLO11Wrapper
        import glob
        candidates = glob.glob(f'{W_BASE}/yolo11/{variant}/**/best.pt', recursive=True)
        wp = candidates[0] if candidates else f'{W_BASE}/yolo11/{variant}/run/weights/best.pt'
        m = YOLO11Wrapper(wp, conf_threshold=conf, variant=variant)
    elif family == 'hrnet':
        from src.models.hrnet_wrapper import HRNetWrapper
        import glob
        pth = glob.glob(f'{W_BASE}/hrnet/{variant}/best*.pth')
        cfg_f = glob.glob(f'{W_BASE}/hrnet/{variant}/*config*.py')
        m = HRNetWrapper(pth[0], cfg_f[0], conf_threshold=conf, variant=variant)
    elif family == 'vitpose':
        from src.models.vitpose_wrapper import ViTPoseWrapper
        import glob
        pth = glob.glob(f'{W_BASE}/vitpose/{variant}/best*.pth')
        cfg_f = glob.glob(f'{W_BASE}/vitpose/{variant}/*config*.py')
        m = ViTPoseWrapper(pth[0], cfg_f[0], conf_threshold=conf, variant=variant)
    elif family == 'detr':
        from src.models.detr_wrapper import DeformableDETRWrapper
        wp = f'{W_BASE}/detr/{variant}/checkpoint_best.pth'
        m = DeformableDETRWrapper(wp, conf_threshold=conf, variant=variant)
    else:
        raise ValueError(f'Unknown family: {family}')
    m.load_model()
    return m

print(f'Loading {BEST_NAME}...')
MODEL = load_model_for_demo(BEST_FAMILY, BEST_VARIANT)
print('Model siap')

In [ ]:
# Cell 6: Helper functions — pitch background + render overlay
from src.geometry.pitch_reference import get_dst_dataframe, PITCH_WIDTH, PITCH_HEIGHT
from src.geometry.homography import compute_homography_from_df

DF_DST = get_dst_dataframe()

CLR_VISIBLE  = (50, 220, 80)
CLR_OCCLUDED = (50, 165, 255)
CLR_PROJ     = (50, 220, 255)
CLR_VALID    = (50, 200, 80)
CLR_INVALID  = (60, 60, 220)


def make_pitch_bg(W, H):
    img = np.full((H, W, 3), (34, 120, 50), dtype=np.uint8)
    def px(cx, cy):
        return int(cx / PITCH_WIDTH * W), int(cy / PITCH_HEIGHT * H)
    def line(p1, p2):
        cv2.line(img, px(*p1), px(*p2), (255,255,255), 1)
    for p1,p2 in [
        ((0,0),(120,0)),((120,0),(120,80)),((120,80),(0,80)),((0,80),(0,0)),
        ((60,0),(60,80)),
        ((0,18),(18,18)),((18,18),(18,62)),((18,62),(0,62)),
        ((120,18),(102,18)),((102,18),(102,62)),((102,62),(120,62)),
        ((0,30),(6,30)),((6,30),(6,50)),((6,50),(0,50)),
        ((120,30),(114,30)),((114,30),(114,50)),((114,50),(120,50)),
    ]:
        line(p1, p2)
    cx, cy = px(60, 40)
    cv2.circle(img, (cx, cy), int(10/PITCH_WIDTH*W), (255,255,255), 1)
    cv2.circle(img, (cx, cy), 3, (255,255,255), -1)
    cv2.circle(img, px(12, 40), 3, (255,255,255), -1)
    cv2.circle(img, px(108, 40), 3, (255,255,255), -1)
    return img


def draw_left(frame, result, show_id):
    out = frame.copy()
    if result is None:
        cv2.putText(out, 'No detection', (20,40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, CLR_INVALID, 2)
        return out
    for row in result.to_dataframe_rows():
        x, y = int(row['x']), int(row['y'])
        clr  = CLR_VISIBLE if row['visibility'] == 2 else CLR_OCCLUDED
        cv2.circle(out, (x,y), 6, clr, -1)
        cv2.circle(out, (x,y), 7, (0,0,0), 1)
        if show_id:
            cv2.putText(out, str(row['kpt_id']), (x+8,y+4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255,255,255), 1)
    return out


def draw_right(pitch_bg, result, homo, PW, PH, sx, sy):
    out   = pitch_bg.copy()
    valid = homo.get('is_H_valid', False) if homo else False
    cv2.rectangle(out, (2,2), (PW-3,PH-3), CLR_VALID if valid else CLR_INVALID, 3)
    if valid and result:
        H_mat = homo['H']
        mask  = homo.get('mask')
        for i, row in enumerate(result.to_dataframe_rows()):
            src = np.array([[[row['x'], row['y']]]], dtype=np.float32)
            dst = cv2.perspectiveTransform(src, H_mat)[0][0]
            px, py = int(dst[0]*sx), int(dst[1]*sy)
            if 0 <= px < PW and 0 <= py < PH:
                is_inlier = mask[i] if mask is not None and i < len(mask) else True
                clr = CLR_PROJ if is_inlier else (80, 80, 200)
                cv2.circle(out, (px,py), 7, clr, -1)
                cv2.circle(out, (px,py), 8, (0,0,0), 1)
        n_in  = homo.get('n_inliers', 0)
        n_out = homo.get('n_outliers', 0)
        cond  = homo.get('condition_number', float('inf'))
        cond_s = f'{cond:.0f}' if cond < 1e6 else 'inf'
        cv2.putText(out, f'Inliers:{n_in}  Out:{n_out}  Cond:{cond_s}',
                    (8, PH-10), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (210,210,210), 1)
    else:
        msg = 'Not enough keypoints' if homo and not homo.get('is_four_points') else 'H invalid / No detection'
        cv2.putText(out, msg, (10, PH//2), cv2.FONT_HERSHEY_SIMPLEX, 0.5, CLR_INVALID, 1)
    return out


def add_hud(canvas, frame_no, total, model_name, n_valid, n_proc, elapsed):
    pct = n_valid / max(n_proc, 1) * 100
    spd = n_proc / max(elapsed, 0.001)
    eta = (total - frame_no) / max(spd, 0.001)
    lines = [
        f'Model  : {model_name}',
        f'Frame  : {frame_no}/{total}',
        f'H valid: {pct:.1f}% ({n_valid}/{n_proc})',
        f'Speed  : {spd:.1f} fps',
        f'ETA    : {eta:.0f}s',
    ]
    ov = canvas.copy()
    cv2.rectangle(ov, (8,8), (290, 16+len(lines)*18), (0,0,0), -1)
    cv2.addWeighted(ov, 0.55, canvas, 0.45, 0, canvas)
    for i, ln in enumerate(lines):
        cv2.putText(canvas, ln, (12, 22+i*18), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (220,220,220), 1)

print('Helper functions loaded')

In [ ]:
# Cell 7: Proses video — JALANKAN INI
def process_video(video_path, model, output_path,
                  process_fps=5, max_seconds=None, output_scale=0.75,
                  show_kpt_id=True, show_stats=True,
                  pitch_w=560, pitch_h=374):

    cap        = cv2.VideoCapture(video_path)
    fps_orig   = cap.get(cv2.CAP_PROP_FPS)
    n_total    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    orig_w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    step       = max(1, int(fps_orig / process_fps)) if process_fps else 1
    max_f      = int(max_seconds * fps_orig) if max_seconds else n_total
    frames_idx = list(range(0, min(n_total, max_f), step))

    out_w_left = int(orig_w * output_scale)
    out_h_left = int(orig_h * output_scale)
    out_h_tot  = max(out_h_left, pitch_h)
    out_w_tot  = out_w_left + pitch_w
    sx         = pitch_w / PITCH_WIDTH
    sy         = pitch_h / PITCH_HEIGHT

    pitch_bg   = make_pitch_bg(pitch_w, pitch_h)
    out_fps    = process_fps if process_fps else fps_orig
    writer     = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'),
                                 out_fps, (out_w_tot, out_h_tot))

    print(f'Processing {len(frames_idx)} frames -> output {out_w_tot}x{out_h_tot} @ {out_fps}fps')

    n_valid, n_proc, t0 = 0, 0, time.time()

    for fi in tqdm(frames_idx, desc='Rendering video'):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret:
            break

        # Inference
        result = model.predict_top1(frame)

        # Homography
        homo = None
        if result:
            rows = result.to_dataframe_rows()
            if rows:
                df_src = pd.DataFrame(rows)
                homo   = compute_homography_from_df(
                    df_src, DF_DST,
                    ransac_threshold=CFG['ransac_threshold'],
                    min_keypoints=CFG['min_keypoints_for_homography'],
                )
                if homo.get('is_H_valid'):
                    n_valid += 1
        n_proc += 1

        # Scale keypoints ke resolusi output untuk panel kiri
        if result:
            result_s = deepcopy(result)
            result_s.keypoints[:, 0] *= (out_w_left / orig_w)
            result_s.keypoints[:, 1] *= (out_h_left / orig_h)
        else:
            result_s = None

        frame_rs  = cv2.resize(frame, (out_w_left, out_h_left))
        left_pan  = draw_left(frame_rs, result_s, show_kpt_id)
        right_pan = draw_right(pitch_bg, result, homo, pitch_w, pitch_h, sx, sy)

        canvas = np.zeros((out_h_tot, out_w_tot, 3), dtype=np.uint8)
        canvas[:out_h_left,  :out_w_left]         = left_pan
        canvas[:pitch_h,     out_w_left:out_w_tot] = right_pan
        cv2.line(canvas, (out_w_left,0), (out_w_left,out_h_tot), (60,60,60), 2)
        cv2.putText(canvas, 'Broadcast view',
                    (10, out_h_tot-10), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150,150,150), 1)
        cv2.putText(canvas, "Bird's-eye view",
                    (out_w_left+10, out_h_tot-10), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150,150,150), 1)

        if show_stats:
            add_hud(canvas, fi, n_total, BEST_NAME, n_valid, n_proc, time.time()-t0)

        writer.write(canvas)

    cap.release()
    writer.release()

    elapsed = time.time() - t0
    size_mb = os.path.getsize(output_path) / 1024**2
    print(f'\nSelesai!')
    print(f'  Output  : {output_path}')
    print(f'  H valid : {n_valid}/{n_proc} ({n_valid/max(n_proc,1)*100:.1f}%)')
    print(f'  Durasi  : {elapsed:.1f}s  |  {size_mb:.1f} MB')
    return output_path


# ── Jalankan ──────────────────────────────────────────────────────────────────
if os.path.exists(VIDEO_INPUT):
    vname       = os.path.splitext(os.path.basename(VIDEO_INPUT))[0]
    OUTPUT_PATH = f'{VIDEO_OUT_DIR}/{vname}_{BEST_FAMILY}_{BEST_VARIANT}_demo.mp4'

    OUTPUT_PATH = process_video(
        video_path=VIDEO_INPUT, model=MODEL, output_path=OUTPUT_PATH,
        process_fps=PROCESS_FPS, max_seconds=MAX_SECONDS,
        output_scale=OUTPUT_SCALE, show_kpt_id=SHOW_KPT_ID, show_stats=SHOW_STATS,
    )
else:
    print('Video tidak ditemukan — isi VIDEO_INPUT di Cell 4')

In [ ]:
# Cell 8: Preview 4 frame dari video hasil
import matplotlib.pyplot as plt

if os.path.exists(OUTPUT_PATH):
    cap_out = cv2.VideoCapture(OUTPUT_PATH)
    total   = int(cap_out.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs    = [0, total//4, total//2, 3*total//4]

    fig, axes = plt.subplots(2, 2, figsize=(18, 9))
    for ax, fi in zip(axes.flatten(), idxs):
        cap_out.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frm = cap_out.read()
        if ret:
            ax.imshow(cv2.cvtColor(frm, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Frame {fi}/{total}', fontsize=9)
        ax.axis('off')
    cap_out.release()

    plt.suptitle(f'Preview: {os.path.basename(OUTPUT_PATH)}', fontsize=12)
    plt.tight_layout()
    prev_path = OUTPUT_PATH.replace('.mp4', '_preview.png')
    plt.savefig(prev_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Preview disimpan: {prev_path}')
    print('Gambar ini bisa masuk ke paper sebagai qualitative result figure')

In [ ]:
# Cell 9: Download video
from google.colab import files
size_mb = os.path.getsize(OUTPUT_PATH) / 1024**2
print(f'File  : {OUTPUT_PATH}')
print(f'Size  : {size_mb:.1f} MB')
print()
print('Video sudah tersimpan di Google Drive.')
print('Untuk download ke komputer, uncomment baris berikut:')
print()
print('# files.download(OUTPUT_PATH)')
# files.download(OUTPUT_PATH)

In [ ]:
# Cell 10: Batch processing (opsional, untuk overnight run)
import glob as gl

VIDEO_FOLDER = '/content/drive/MyDrive/Colab Notebooks/Videos/'
video_files  = (gl.glob(f'{VIDEO_FOLDER}/*.mp4') +
                gl.glob(f'{VIDEO_FOLDER}/*.avi') +
                gl.glob(f'{VIDEO_FOLDER}/*.mkv'))

print(f'{len(video_files)} video ditemukan di {VIDEO_FOLDER}')
for vf in video_files:
    print(f'  - {os.path.basename(vf)}')

# Uncomment untuk batch process:
# for vf in video_files:
#     vname = os.path.splitext(os.path.basename(vf))[0]
#     out   = f'{VIDEO_OUT_DIR}/{vname}_{BEST_FAMILY}_{BEST_VARIANT}_demo.mp4'
#     print(f'Processing: {vname}')
#     process_video(vf, MODEL, out, process_fps=5, max_seconds=60,
#                   output_scale=0.75, show_stats=True)

In [ ]:
# Cell 11: Unload model
import gc
MODEL.unload()
del MODEL
gc.collect()
torch.cuda.empty_cache()
print('Model unloaded — GPU memory freed')